# Tratamento da PPH 2019 para regressão de consumo energético

Este notebook transforma a aba **Banco de Dados** da PPH 2019 em uma base analítica com **uma linha por residência (`ENTREVISTA`)**. O foco é preservar características do domicílio, iluminação, principais aparelhos e o histórico mensal da conta de energia para uso posterior em modelos de regressão.

O arquivo original não é alterado. É gerada uma base unificada, acompanhada por um dicionário das variáveis.

## Convenções adotadas

- `P5.1` não filtra registros: o tipo de domicílio é mantido como feature categórica para posterior one-hot encoding.
- Os registros de `LOOP` são aparelhos adicionais da mesma entrevista. Eles são agregados, nunca descartados de forma indiscriminada.
- Frequências categóricas são convertidas para vezes por semana pelos pontos médios: 6,5; 4,5; 2,5; 1; 0,5; 0,25; e 0.
- No micro-ondas, as faixas de duração usam seus pontos médios; **acima de 90 minutos = 120 minutos**.
- No banho, as faixas usam pontos médios e **acima de 20 minutos = 25 minutos** para construir um proxy conservador.
- Iluminação considera somente o número de lâmpadas. Nenhuma duração ou potência estimada é usada.
- A temperatura de 2019 é resumida em uma única média anual por UF. Primeiro calcula-se a média de cada estação e depois a média entre estações, para que todas tenham o mesmo peso.
- Ausências estruturais de quantidades (por exemplo, uma residência sem freezer) viram zero. Ausências informativas não são confundidas automaticamente com zero.
- A conta de luz é resumida pela **mediana dos 12 meses válidos**; meses ausentes ou valores não positivos não entram na mediana. Se nenhum mês for válido, o alvo recebe zero e uma flag identifica esse caso.
- Produtos entre frequência e tempo são chamados de **proxy de uso**, não de consumo em kWh, pois a pesquisa não fornece potência elétrica nominal para todos os aparelhos.


## 1. Bibliotecas e localização do arquivo

O bloco abaixo importa as bibliotecas e procura o XLSX tanto na pasta do projeto quanto na pasta pai do notebook. Assim, o notebook funciona quando executado a partir da raiz ou de `equipe-dados`.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

NOME_XLSX = "PPH 2019 - Banco de Dados V2.xlsx"
candidatos = [Path(NOME_XLSX), Path("..") / NOME_XLSX]
ARQUIVO_XLSX = next((p.resolve() for p in candidatos if p.exists()), None)
if ARQUIVO_XLSX is None:
    raise FileNotFoundError(f"Não foi possível localizar {NOME_XLSX!r} em {candidatos}.")

NOME_CLIMA = "2019.csv"
candidatos_clima = [Path(NOME_CLIMA), Path("..") / NOME_CLIMA]
ARQUIVO_CLIMA = next((p.resolve() for p in candidatos_clima if p.exists()), None)
if ARQUIVO_CLIMA is None:
    raise FileNotFoundError(f"Não foi possível localizar {NOME_CLIMA!r} em {candidatos_clima}.")

PASTA_SAIDA = Path("equipe-dados") if Path("equipe-dados").is_dir() else Path(".")
ARQUIVO_SAIDA = PASTA_SAIDA / "dados_pph2019_tratados_regressao.csv"
ARQUIVO_DICIONARIO = PASTA_SAIDA / "dicionario_dados_pph2019_tratados.csv"

print(f"Fonte: {ARQUIVO_XLSX}")
print(f"Clima: {ARQUIVO_CLIMA}")
print(f"Saída unificada: {ARQUIVO_SAIDA.resolve()}")


## 2. Seleção explícita das colunas brutas

A planilha tem 2.319 colunas. Aqui são selecionados apenas os campos necessários. Os blocos de iluminação são localizados por expressão regular, evitando listas manuais frágeis.

Os horários de ar-condicionado, TV e máquina de lavar são indicadores de faixas horárias selecionadas. Eles são usados apenas para construir proxies de intensidade de uso; para lâmpadas, conforme a convenção definida, os horários não são importados.


In [ ]:
cabecalho = pd.read_excel(ARQUIVO_XLSX, sheet_name="Banco de Dados", nrows=0)
todas_colunas = cabecalho.columns.astype(str).tolist()

colunas_base = [
    "ENTREVISTA", "LOOP", "UF", "MUNICIPIO",
    "P4.1_1", "P5.1", "P5.3", "P5.7", "P5.7_1_TXT", "P5.13",
]
colunas_conta = [f"P5.15.2_{mes}" for mes in range(1, 13)]

# Quantidades de lâmpadas: 12 ambientes/modos x 26 tipos.
regex_lampadas = re.compile(r"^P6\.1\.2\.\d+_\d+_TXT$")
colunas_lampadas = [c for c in todas_colunas if regex_lampadas.fullmatch(c)]

colunas_aparelhos = [
    # Refrigeradores e freezers
    "P7.1", "P7.1.1.2", "P7.1.1.3", "P7.1.1.4", "P7.1.1.7", "P7.1.1.5_1_TXT", "P7.1.1.6",
    "P7.2", "P7.2.1.2", "P7.2.1.3", "P7.2.1.4", "P7.2.1.7", "P7.2.1.5_1_TXT", "P7.2.1.6",
    # Ar-condicionado
    "P7.3", "P7.3.1.2", "P7.3.1.3", "P7.3.1.4", "P7.3.1.5", "P7.3.1.6",
    "P7.3.1.7", "P7.3.1.8_1_TXT", "P7.3.2.5", "P7.3.3",
    # Televisores
    "P8.1", "P8.1.1.2.1", "P8.1.1.2.2", "P8.1.1.3_1_TXT", "P8.1.1.4", "P8.1.1.5",
    # Micro-ondas
    "P8.2", "P8.2.1.2", "P8.2.1.4", "P8.2.1.3", "P8.2.1.5_1_TXT", "P8.2.1.6",
    # Máquinas de lavar
    "P8.3", "P8.3.1.2", "P8.3.1.3", "P8.3.1.5_1_TXT", "P8.3.1.6",
    "P8.3.1.7_1_TXT", "P8.3.1.8",
    # Chuveiros
    "P10.8", "P10.9.2", "P10.9.3", "P10.9.4", "P10.9.5", "P10.9.D",
]

colunas_horas_ar = [f"P7.3.2.2_{h}" for h in range(24)]
colunas_freq_mensal_ar = [f"P7.3.2.4.{mes}" for mes in range(1, 13)]
colunas_horas_tv = [f"P8.1.1.2_{h}" for h in range(24)]
colunas_horas_lavadora = [f"P8.3.1.4_{h}" for h in range(24)]

colunas_selecionadas = list(dict.fromkeys(
    colunas_base + colunas_conta + colunas_lampadas + colunas_aparelhos
    + colunas_horas_ar + colunas_freq_mensal_ar
    + colunas_horas_tv + colunas_horas_lavadora
))

faltantes = sorted(set(colunas_selecionadas) - set(todas_colunas))
if faltantes:
    raise KeyError(f"Colunas esperadas não encontradas: {faltantes}")

print(f"Colunas selecionadas: {len(colunas_selecionadas)} de {len(todas_colunas)}")
print(f"Colunas de quantidades de lâmpadas: {len(colunas_lampadas)}")


## 3. Leitura eficiente do XLSX

Mesmo selecionando poucas colunas, leitores convencionais percorrem as mais de 60 milhões de posições da planilha. A função abaixo lê diretamente o XML interno do XLSX e materializa somente as colunas selecionadas. O resultado continua sendo um `DataFrame` pandas comum.

Essa otimização muda apenas a forma de leitura; não altera valores nem regras de negócio.


In [ ]:
def _numero_coluna_excel(letras: str) -> int:
    numero = 0
    for letra in letras:
        numero = numero * 26 + ord(letra) - 64
    return numero


def ler_xlsx_colunas(arquivo: Path, colunas: list[str]) -> pd.DataFrame:
    try:
        from lxml import etree
    except ImportError as exc:
        raise ImportError("Instale 'lxml' para executar a leitura otimizada.") from exc

    ns = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    regex_ref = re.compile(r"([A-Z]+)\d+")
    desejadas = set(colunas)

    def shared_strings(zip_file):
        valores = []
        with zip_file.open("xl/sharedStrings.xml") as stream:
            for _, elemento in etree.iterparse(stream, events=("end",), tag=ns + "si"):
                textos = elemento.xpath(".//main:t/text()", namespaces={"main": ns[1:-1]})
                valores.append("".join(textos))
                elemento.clear()
                while elemento.getprevious() is not None:
                    del elemento.getparent()[0]
        return valores

    def valor_celula(celula, textos_compartilhados):
        tipo = celula.get("t")
        if tipo == "inlineStr":
            textos = celula.xpath(".//main:t/text()", namespaces={"main": ns[1:-1]})
            return "".join(textos)
        valor = celula.find(ns + "v")
        if valor is None or valor.text is None:
            return None
        if tipo == "s":
            return textos_compartilhados[int(valor.text)]
        if tipo == "b":
            return int(valor.text) == 1
        return valor.text

    registros = []
    with ZipFile(arquivo) as zip_file:
        textos_compartilhados = shared_strings(zip_file)
        indice_para_nome = {}
        with zip_file.open("xl/worksheets/sheet1.xml") as stream:
            for _, linha in etree.iterparse(stream, events=("end",), tag=ns + "row"):
                numero_linha = int(linha.get("r"))
                if numero_linha == 1:
                    for celula in linha.findall(ns + "c"):
                        letras = regex_ref.match(celula.get("r")).group(1)
                        indice = _numero_coluna_excel(letras)
                        nome = valor_celula(celula, textos_compartilhados)
                        if nome in desejadas:
                            indice_para_nome[indice] = nome
                    nao_encontradas = desejadas - set(indice_para_nome.values())
                    if nao_encontradas:
                        raise KeyError(f"Colunas não encontradas: {sorted(nao_encontradas)}")
                else:
                    registro = {nome: None for nome in colunas}
                    for celula in linha.findall(ns + "c"):
                        letras = regex_ref.match(celula.get("r")).group(1)
                        nome = indice_para_nome.get(_numero_coluna_excel(letras))
                        if nome is not None:
                            registro[nome] = valor_celula(celula, textos_compartilhados)
                    registros.append(registro)
                linha.clear()
                while linha.getprevious() is not None:
                    del linha.getparent()[0]
    return pd.DataFrame(registros, columns=colunas)


df_bruto = ler_xlsx_colunas(ARQUIVO_XLSX, colunas_selecionadas)
df_bruto["ENTREVISTA"] = pd.to_numeric(df_bruto["ENTREVISTA"], errors="raise").astype("int64")
df_bruto["LOOP"] = pd.to_numeric(df_bruto["LOOP"], errors="raise").astype("int16")

print(f"Linhas brutas: {len(df_bruto):,}")
print(f"Entrevistas únicas: {df_bruto['ENTREVISTA'].nunique():,}")
df_bruto.head(3)


## 4. Funções de agregação e conversões

As funções abaixo concentram as regras reutilizadas:

- `primeiro_valido`: recupera o valor domiciliar preenchido no primeiro loop válido;
- `contar_categorias`: converte características de vários aparelhos em quantidades por categoria, sem perder o segundo ou terceiro aparelho;
- `frequencia_semanal`: traduz as classes de frequência pelos pontos médios acordados;
- `quantidade_selecoes`: conta faixas horárias marcadas em perguntas de múltipla seleção.

Os nomes resultantes já são interpretáveis e não expõem códigos `P.x.x` na base final.


In [ ]:
def numerico(serie: pd.Series) -> pd.Series:
    return pd.to_numeric(serie, errors="coerce")


def primeiro_valido(serie: pd.Series):
    valores = serie.dropna()
    return valores.iloc[0] if len(valores) else np.nan


MAPA_FREQUENCIA = {1: 6.5, 2: 4.5, 3: 2.5, 4: 1.0, 5: 0.5, 6: 0.25, 7: 0.0}


def frequencia_semanal(serie: pd.Series) -> pd.Series:
    return numerico(serie).map(MAPA_FREQUENCIA)


def quantidade_selecoes(df: pd.DataFrame, colunas: list[str]) -> pd.Series:
    return df[colunas].notna().sum(axis=1).astype(float)


def contar_categorias(df: pd.DataFrame, coluna: str, prefixo: str, mapa: dict) -> pd.DataFrame:
    codigos = numerico(df[coluna])
    partes = []
    for codigo, rotulo in mapa.items():
        contagem = codigos.eq(codigo).groupby(df["ENTREVISTA"]).sum().astype("int16")
        partes.append(contagem.rename(f"{prefixo}_{rotulo}"))
    return pd.concat(partes, axis=1)


def media_por_entrevista(df: pd.DataFrame, serie: pd.Series) -> pd.Series:
    return serie.groupby(df["ENTREVISTA"]).mean()


def soma_por_entrevista(df: pd.DataFrame, serie: pd.Series) -> pd.Series:
    return serie.groupby(df["ENTREVISTA"]).sum(min_count=1)


## 5. Consolidação domiciliar, conta de luz e iluminação

Este bloco inicia a base final no índice `ENTREVISTA` e consolida os atributos do domicílio. Como esses atributos aparecem apenas em um loop, usa-se o primeiro valor não nulo.

Para a conta de luz, os doze meses permanecem disponíveis com nomes claros e também são criadas a mediana, a quantidade de meses válidos e uma flag de ausência total. A mediana é robusta a picos mensais e não é reduzida artificialmente por células vazias.

As quantidades de lâmpadas são somadas entre todos os tipos e cômodos em uma única feature, `numero_total_lampadas`. Células vazias de quantidade são interpretadas como zero; duração e potência não participam do cálculo.


In [ ]:
ids = pd.Index(sorted(df_bruto["ENTREVISTA"].unique()), name="id_entrevista")
df_final = pd.DataFrame(index=ids)

# Atributos domiciliares: primeiro valor preenchido entre os loops da entrevista.
base_origem = {
    "UF": "uf",
    "MUNICIPIO": "municipio",
    "P4.1_1": "numero_moradores",
    "P5.1": "tipo_domicilio",
    "P5.3": "cor_paredes_externas",
    "P5.13": "tipo_uso_domicilio",
}
for origem, destino in base_origem.items():
    df_final[destino] = df_bruto.groupby("ENTREVISTA")[origem].agg(primeiro_valido).reindex(ids)

df_final["tipo_domicilio"] = numerico(df_final["tipo_domicilio"]).map({
    1: "casa", 2: "apartamento", 3: "quarto_ou_comodo",
}).fillna("nao_informado")
df_final["cor_paredes_externas"] = numerico(df_final["cor_paredes_externas"]).map({
    1: "clara", 2: "media", 3: "escura",
}).fillna("nao_informado")
df_final["tipo_uso_domicilio"] = numerico(df_final["tipo_uso_domicilio"]).map({
    1: "exclusivamente_residencial", 2: "possui_atividade_comercial_ou_industrial", 9: "nao_informado",
}).fillna("nao_informado")
df_final["numero_moradores"] = numerico(df_final["numero_moradores"]).fillna(0).clip(lower=0)

# A área efetiva está em P5.7_1_TXT; P5.7 é somente o indicador de resposta.
area = df_bruto.groupby("ENTREVISTA")["P5.7_1_TXT"].agg(primeiro_valido).reindex(ids)
area = numerico(area).where(lambda s: s.gt(0))
df_final["area_construida_m2"] = area.fillna(0)

# Conta mensal: valores ausentes e não positivos não entram na mediana.
nomes_meses = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
conta_linhas = df_bruto[colunas_conta].apply(pd.to_numeric, errors="coerce")
conta_linhas = conta_linhas.mask(conta_linhas <= 0)
conta_por_entrevista = conta_linhas.groupby(df_bruto["ENTREVISTA"]).agg(primeiro_valido).reindex(ids)
conta_por_entrevista.columns = [f"consumo_kwh_{mes}" for mes in nomes_meses]
df_final = df_final.join(conta_por_entrevista)
df_final["quantidade_meses_conta_validos"] = conta_por_entrevista.notna().sum(axis=1).astype("int8")
df_final["conta_luz_mediana_kwh_ausente"] = conta_por_entrevista.notna().sum(axis=1).eq(0).astype("int8")
df_final["conta_luz_mediana_kwh"] = conta_por_entrevista.median(axis=1, skipna=True).fillna(0)

# A base de modelagem usa a mediana; os meses são preservados para auditoria e análises temporais.
for coluna in conta_por_entrevista.columns:
    df_final[coluna] = df_final[coluna].fillna(0)

ambientes = {
    1: "quarto_uso_habitual", 2: "quarto_uso_eventual",
    3: "banheiro_uso_habitual", 4: "banheiro_uso_eventual",
    5: "sala", 6: "cozinha", 7: "area_servico", 8: "garagem",
    9: "area_externa", 10: "corredores", 11: "varandas", 12: "outros_ambientes",
}
lampadas_numericas = df_bruto[colunas_lampadas].apply(pd.to_numeric, errors="coerce").fillna(0).clip(lower=0)
# Soma direta de todos os tipos e cômodos; não são mantidas features por ambiente.
lampadas_total_por_linha = lampadas_numericas.sum(axis=1)
df_final["numero_total_lampadas"] = (
    lampadas_total_por_linha.groupby(df_bruto["ENTREVISTA"]).max().reindex(ids).fillna(0)
)

df_final.head(3)


## 6. Integração da temperatura média de 2019 por UF

O arquivo climático possui mais de cinco milhões de medições horárias. Ele é lido em blocos para controlar o uso de memória.

A média é calculada em dois níveis: primeiro por estação meteorológica e depois entre as estações de cada UF. Essa ordem impede que uma estação com mais horas preenchidas receba mais peso apenas por ter menos dados ausentes. O resultado é uma única feature, `temperatura_media_2019_uf_c`, integrada à residência por `uf`.


In [ ]:
COLUNA_TEMPERATURA = "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)"
parciais_estacoes = []

for bloco in pd.read_csv(
    ARQUIVO_CLIMA,
    usecols=["UF", "CODIGO (WMO)", COLUNA_TEMPERATURA],
    decimal=",",
    chunksize=300_000,
    low_memory=False,
):
    bloco[COLUNA_TEMPERATURA] = pd.to_numeric(bloco[COLUNA_TEMPERATURA], errors="coerce")
    parcial = bloco.groupby(["UF", "CODIGO (WMO)"])[COLUNA_TEMPERATURA].agg(["sum", "count"])
    parciais_estacoes.append(parcial)

acumulado_estacoes = pd.concat(parciais_estacoes).groupby(level=[0, 1]).sum()
acumulado_estacoes = acumulado_estacoes[acumulado_estacoes["count"] > 0]
acumulado_estacoes["temperatura_media_estacao_c"] = (
    acumulado_estacoes["sum"] / acumulado_estacoes["count"]
)

temperatura_media_uf = (
    acumulado_estacoes["temperatura_media_estacao_c"]
    .groupby(level="UF")
    .mean()
    .rename("temperatura_media_2019_uf_c")
)

df_final["temperatura_media_2019_uf_c"] = df_final["uf"].map(temperatura_media_uf)

ufs_sem_temperatura = sorted(set(df_final.loc[df_final["temperatura_media_2019_uf_c"].isna(), "uf"]))
if ufs_sem_temperatura:
    raise ValueError(f"UFs sem temperatura média de 2019: {ufs_sem_temperatura}")

print(f"UFs climáticas integradas: {temperatura_media_uf.size}")
display(temperatura_media_uf.sort_index().to_frame())


## 7. Quantidades, características e proxies de uso dos aparelhos

As quantidades declaradas (`P7.1`, `P7.2`, `P7.3`, `P8.1`, `P8.2`, `P8.3` e `P10.8`) são atributos domiciliares e usam o primeiro valor válido. Elas geram uma única variável `numero_total_*` para cada classe de aparelho. As colunas de tipo, capacidade, função e tamanho não são mantidas na base modelável: elas descrevem dimensões diferentes dos mesmos itens e somá-las conjuntamente duplicaria a contagem. Além disso, a pergunta direta de total é mais completa do que algumas respostas de capacidade.

- idades, temperaturas e outras medidas numéricas viram médias entre os aparelhos informados;
- frequência × tempo é calculada aparelho a aparelho e depois somada na residência;
- horários marcados são tratados como número de faixas de uma hora e, por isso, as variáveis recebem o sufixo `_proxy`.

### O que significa proxy

Uma proxy é uma aproximação construída com as respostas disponíveis; não é uma medição direta. Assim, `horas_marcadas_*_proxy`, `minutos_microondas_semana_proxy` e `numero_lavagens_semana_proxy` representam intensidade relativa de uso, mas não consumo elétrico medido. Apenas o proxy dos chuveiros chega a kWh/dia, porque combina potência, duração e quantidade de banhos informadas.

`numero_banhos_dia` não foi inventada: vem de `P10.9.5`, registrada nas linhas de cada chuveiro, e é somada por residência. A frequência categórica é convertida em vezes por semana pelos pontos médios `6,5`, `4,5`, `2,5`, `1`, `0,5`, `0,25` e `0`; para ar-condicionado, calcula-se primeiro a média dessas frequências nos 12 meses.

`interacao_ar_condicionado_frequencia` é `numero_total_ar_condicionados × frequencia_media_ar_condicionado_vezes_semana`, aproximando conjuntamente posse e intensidade de uso.

Refrigeradores e freezers não têm duração diária explícita no questionário. Para eles são mantidas quantidade, frequência e idade, sem inventar horas de uso.


In [ ]:
# Totais de aparelhos informados diretamente no questionário. Não se somam tipo e
# capacidade, pois são duas classificações do mesmo aparelho e isso duplicaria a contagem.
# Ausência estrutural é convertida em zero.
mapa_quantidades = {
    "P7.1": "numero_total_refrigeradores",
    "P7.2": "numero_total_freezers",
    "P7.3": "numero_total_ar_condicionados",
    "P8.1": "numero_total_televisores",
    "P8.2": "numero_total_microondas",
    "P8.3": "numero_total_maquinas_lavar",
    "P10.8": "numero_total_chuveiros",
}
for origem, destino in mapa_quantidades.items():
    serie = df_bruto.groupby("ENTREVISTA")[origem].agg(primeiro_valido).reindex(ids)
    df_final[destino] = numerico(serie).fillna(0).clip(lower=0)

df_final["numero_total_aparelhos_selecionados"] = df_final[list(mapa_quantidades.values())].sum(axis=1)

# Chuveiro: características intermediárias e proxy técnico de consumo diário.
potencia_chuveiro_w = numerico(df_bruto["P10.9.4"]).map({1:4000.0, 2:5250.0, 3:6750.0, 4:8000.0})
duracao_banho_min = numerico(df_bruto["P10.9.D"]).map({1:5.0, 2:8.0, 3:15.5, 4:25.0})
banhos_dia = numerico(df_bruto["P10.9.5"]).where(lambda s: s.ge(0))
fonte_eletrica = numerico(df_bruto["P10.9.2"]).eq(1)

df_final["potencia_media_chuveiros_eletricos_w"] = media_por_entrevista(
    df_bruto, potencia_chuveiro_w.where(fonte_eletrica)
).reindex(ids).fillna(0)
df_final["duracao_media_banho_min"] = media_por_entrevista(
    df_bruto, duracao_banho_min
).reindex(ids).fillna(0)
df_final["numero_banhos_dia"] = soma_por_entrevista(df_bruto, banhos_dia).reindex(ids).fillna(0)
df_final["consumo_chuveiros_kwh_dia_proxy"] = soma_por_entrevista(
    df_bruto,
    (potencia_chuveiro_w / 1000) * (duracao_banho_min / 60) * banhos_dia * fonte_eletrica.astype(float),
).reindex(ids).fillna(0)

# Tipo, capacidade e tamanho não entram na base final: cada dimensão descreve os
# mesmos aparelhos e sua soma conjunta produziria dupla contagem.
# Idade média dos aparelhos; zero somente quando não há valor informado.
idades = {
    "P7.1.1.5_1_TXT": "idade_media_refrigeradores_anos",
    "P7.2.1.5_1_TXT": "idade_media_freezers_anos",
    "P7.3.1.8_1_TXT": "idade_media_ar_condicionados_anos",
    "P8.1.1.3_1_TXT": "idade_media_televisores_anos",
    "P8.2.1.5_1_TXT": "idade_media_microondas_anos",
    "P8.3.1.7_1_TXT": "idade_media_maquinas_lavar_anos",
}
for origem, destino in idades.items():
    valores = numerico(df_bruto[origem]).where(lambda s: s.ge(0))
    df_final[destino] = media_por_entrevista(df_bruto, valores).reindex(ids).fillna(0)

# Frequências de refrigeradores e freezers: soma das frequências equivalentes dos aparelhos.
df_final["frequencia_total_refrigeradores_vezes_semana"] = soma_por_entrevista(
    df_bruto, frequencia_semanal(df_bruto["P7.1.1.4"])
).reindex(ids).fillna(0)
df_final["frequencia_total_freezers_vezes_semana"] = soma_por_entrevista(
    df_bruto, frequencia_semanal(df_bruto["P7.2.1.4"])
).reindex(ids).fillna(0)

# Ar-condicionado: média anual da frequência mensal e número de faixas horárias marcadas.
freq_ar_meses = pd.concat(
    [frequencia_semanal(df_bruto[c]) for c in colunas_freq_mensal_ar], axis=1
)
freq_ar_media = freq_ar_meses.mean(axis=1, skipna=True).fillna(0)
horas_ar_proxy = quantidade_selecoes(df_bruto, colunas_horas_ar)
df_final["frequencia_media_ar_condicionado_vezes_semana"] = media_por_entrevista(
    df_bruto, freq_ar_media
).reindex(ids).fillna(0)
df_final["horas_marcadas_ar_condicionado_semana_proxy"] = soma_por_entrevista(
    df_bruto, freq_ar_media * horas_ar_proxy
).reindex(ids).fillna(0)
temperatura_ar = numerico(df_bruto["P7.3.1.6"]).where(lambda s: s.between(16, 30))
df_final["temperatura_media_ar_condicionado_c"] = media_por_entrevista(
    df_bruto, temperatura_ar
).reindex(ids).fillna(0)

# TV: frequência semanal vezes quantidade de faixas horárias informadas.
freq_tv = frequencia_semanal(df_bruto["P8.1.1.4"]).fillna(0)
horas_tv_proxy = quantidade_selecoes(df_bruto, colunas_horas_tv)
df_final["horas_marcadas_televisores_semana_proxy"] = soma_por_entrevista(
    df_bruto, freq_tv * horas_tv_proxy
).reindex(ids).fillna(0)

# Micro-ondas: ponto médio das faixas em minutos; faixa aberta recebe 120 minutos.
mapa_tempo_micro_min = {1: 5.0, 2: 20.5, 3: 45.5, 4: 75.5, 5: 120.0}
tempo_micro_min = numerico(df_bruto["P8.2.1.3"]).map(mapa_tempo_micro_min)
freq_micro = frequencia_semanal(df_bruto["P8.2.1.4"])
df_final["minutos_microondas_semana_proxy"] = soma_por_entrevista(
    df_bruto, freq_micro * tempo_micro_min
).reindex(ids).fillna(0)

# Máquina de lavar: frequência x lavagens por dia de uso. Horários são mantidos à parte.
lavagens_por_uso = numerico(df_bruto["P8.3.1.5_1_TXT"]).where(lambda s: s.ge(0))
freq_lavadora = frequencia_semanal(df_bruto["P8.3.1.6"])
df_final["numero_lavagens_semana_proxy"] = soma_por_entrevista(
    df_bruto, freq_lavadora * lavagens_por_uso
).reindex(ids).fillna(0)
df_final["numero_faixas_horarias_maquinas_lavar"] = soma_por_entrevista(
    df_bruto, quantidade_selecoes(df_bruto, colunas_horas_lavadora)
).reindex(ids).fillna(0)

# Interação entre posse e frequência de uso do ar-condicionado.
df_final["interacao_ar_condicionado_frequencia"] = (
    df_final["numero_total_ar_condicionados"]
    * df_final["frequencia_media_ar_condicionado_vezes_semana"]
)

df_final.head(3)


## 8. Integridade, tipos e exportação

As verificações abaixo são condições obrigatórias da saída:

1. uma única linha por `id_entrevista`;
2. mesma quantidade de linhas que entrevistas únicas na fonte;
3. nenhuma coluna com código bruto `P.x.x`;
4. nenhuma quantidade negativa;
5. mediana anual coerente com os meses válidos.

As categorias permanecem como texto para que o pipeline de treinamento aplique one-hot encoding sem tratar códigos numéricos como variáveis ordinais.


In [ ]:
df_final = df_final.reset_index()

# Completa contagens geradas por categoria e compacta colunas numéricas.
colunas_numero = [c for c in df_final.columns if c.startswith("numero_")]
for coluna in colunas_numero:
    df_final[coluna] = pd.to_numeric(df_final[coluna], errors="coerce").fillna(0)

assert df_final["id_entrevista"].is_unique, "Existem entrevistas duplicadas após a consolidação."
assert len(df_final) == df_bruto["ENTREVISTA"].nunique(), "Alguma entrevista foi perdida ou duplicada."
assert not any(re.match(r"^P\d", c) for c in df_final.columns), "Restaram nomes de colunas não interpretáveis."
assert (df_final[colunas_numero] >= 0).all().all(), "Foi encontrada quantidade negativa."
prefixos_detalhados = (
    "numero_refrigeradores_tipo_", "numero_refrigeradores_capacidade_",
    "numero_freezers_tipo_", "numero_freezers_capacidade_",
    "numero_ar_condicionados_tipo_", "numero_ar_condicionados_capacidade_",
    "numero_televisores_tela_", "numero_televisores_tamanho_",
    "numero_microondas_capacidade_", "numero_maquinas_lavar_tipo_",
    "numero_maquinas_lavar_capacidade_", "numero_chuveiros_fonte_",
    "numero_chuveiros_tipo_",
)
assert not any(c.startswith(prefixos_detalhados) for c in df_final.columns), "Restaram contagens desagregadas de aparelhos."
assert df_final["temperatura_media_2019_uf_c"].notna().all(), "Existem residências sem temperatura média da UF."
assert df_final["temperatura_media_2019_uf_c"].between(0, 40).all(), "Temperatura média estadual fora da faixa plausível."

# Validação independente da mediana para uma amostra determinística.
amostra_ids = df_final["id_entrevista"].sample(min(100, len(df_final)), random_state=42)
checagem = df_final.set_index("id_entrevista").loc[amostra_ids]
mediana_recalculada = checagem[[f"consumo_kwh_{m}" for m in nomes_meses]].replace(0, np.nan).median(axis=1).fillna(0)
assert np.allclose(mediana_recalculada, checagem["conta_luz_mediana_kwh"]), "Mediana anual inconsistente."

# As contas mensais são retiradas da base de treinamento para evitar vazamento do alvo.
colunas_mensais = [f"consumo_kwh_{m}" for m in nomes_meses]
df_modelo_completo = df_final.drop(columns=colunas_mensais).rename(columns={
    "quantidade_meses_conta_validos": "quantidade_meses_alvo_validos",
    "conta_luz_mediana_kwh_ausente": "alvo_consumo_ausente",
    "conta_luz_mediana_kwh": "alvo_mediana_consumo_kwh",
})

# Mantém somente residências cujo alvo anual se apoia em pelo menos seis meses.
df_modelo = (
    df_modelo_completo.loc[df_modelo_completo["quantidade_meses_alvo_validos"] >= 6]
    .drop(columns="alvo_consumo_ausente")
    .reset_index(drop=True)
)
assert df_modelo["id_entrevista"].is_unique, "IDs duplicados na base unificada."
assert not any(c.startswith("consumo_kwh_") for c in df_modelo.columns), "Vazamento mensal na base unificada."
assert df_modelo["quantidade_meses_alvo_validos"].ge(6).all(), "Há residências com menos de seis meses válidos."

df_modelo.to_csv(ARQUIVO_SAIDA, index=False, encoding="utf-8")

print(f"Base unificada: {df_modelo.shape[0]:,} residências x {df_modelo.shape[1]} variáveis")
print(f"Duplicidades: {df_modelo['id_entrevista'].duplicated().sum()}")
print(f"Arquivo salvo em: {ARQUIVO_SAIDA.resolve()}")
df_modelo.head()
